# Vishwakarma Institute of Technology, Pune
## Course: CI3201 - Deep Learning
### Lab Practical 5: Comparing RNN, LSTM, and GRU Models for Sequence Classification

---

## 1. Objective
Implement and compare standard Recurrent Neural Networks (RNN), Long Short-Term Memory (LSTM), and Gated Recurrent Units (GRU) for sequence classification on the Amazon Alexa Reviews dataset (sentiment analysis).

In [ ]:
import os
import re
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 1. Load Dataset
df = pd.read_csv('Datasets/amazon_alexa.tsv', sep='\t')
df = df.dropna(subset=['verified_reviews', 'feedback'])
print(df['feedback'].value_counts())

# 2. Clean Text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

df['cleaned_reviews'] = df['verified_reviews'].apply(clean_text)
reviews = df['cleaned_reviews'].values
labels = df['feedback'].values

In [ ]:
# 3. Build Vocabulary
words = [word for review in reviews for word in review.split()]
counter = Counter(words)
vocab = {word: i+1 for i, (word, _) in enumerate(counter.most_common())}
vocab_size = len(vocab) + 1

# 4. Tokenize and Pad
max_len = 50
encoded_reviews = []
for review in reviews:
    encoded = [vocab[word] for word in review.split() if word in vocab]
    if len(encoded) < max_len:
        encoded = [0]*(max_len - len(encoded)) + encoded  # Pre-pad
    else:
        encoded = encoded[:max_len]  # Truncate
    encoded_reviews.append(encoded)

X = np.array(encoded_reviews)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Create DataLoaders
train_data = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).float())
test_data = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test).float())

batch_size = 64
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=False, batch_size=batch_size)

In [ ]:
# 6. Define Models
class SequenceClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, model_type='RNN'):
        super(SequenceClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.model_type = model_type
        
        if model_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        elif model_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
            
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, hidden = self.rnn(embedded)
        
        if self.model_type == 'LSTM':
            hidden = hidden[0]  # LSTM returns (h_n, c_n)
            
        hidden = hidden.squeeze(0)
        output = self.fc(hidden)
        return self.sigmoid(output).squeeze()

embed_dim = 64
hidden_dim = 128

In [ ]:
def train_model(model, train_loader, test_loader, epochs=5):
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    history = {'train_acc': [], 'test_acc': [], 'time': 0}
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        correct, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(inputs)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            
            preds = (output >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        history['train_acc'].append(correct / total)
        
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                output = model(inputs)
                preds = (output >= 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        history['test_acc'].append(correct / total)
        
    history['time'] = time.time() - start_time
    return history

In [ ]:
# Train and Compare All 3 Models
models = ['RNN', 'LSTM', 'GRU']
results = {}

for m_type in models:
    print(f"Training {m_type}...")
    model = SequenceClassifier(vocab_size, embed_dim, hidden_dim, model_type=m_type)
    results[m_type] = train_model(model, train_loader, test_loader, epochs=10)
    print(f"{m_type} trained in {results[m_type]['time']:.2f}s | Final Test Acc: {results[m_type]['test_acc'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
for m_type in models:
    plt.plot(results[m_type]['test_acc'], label=f'{m_type} (Test Acc)')
plt.title('Validation Accuracy Comparison: RNN vs LSTM vs GRU')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()